## 21. Higher Order Functions — objective

- Distinguish query results from materialized table state.
- Follow nested JSON and arrays as they become columns or rows.

Historical results; no code executed.

# 21. Higher Order Functions

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea03-etl-with-apache-spark/21.%20Higher%20Order%20Functions.json)

## Higher Order Functions

  > - Higher Order Functions are functions that operate on complex data types such as arrays and maps
  > - They allow you to pass functions as arguments (such as lambda expressions), apply transformations and return arrays or maps 
  > - They are extremely useful for manipulating arrays without exploding them. 


#### Commonly used Higher Order Array Functions  
  - TRANSFORM
  - FILTER
  - EXISTS
  - AGGREGATE    

#### Syntax
-----------------------------------------------------------------------------
`<function_name> (array_column, lambda_expression)` 

_lambda_expression_: `element -> expression`


Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TEMPORARY VIEW order_items AS
SELECT * FROM 
VALUES
  (1, array('smartphone', 'laptop', 'monitor')),
  (2, array('tablet', 'headphones', 'smartwatch')),
  (3, array('keyboard', 'mouse'))
AS orders(order_id, items);
```

In [ ]:
# Archived output only; no computation.

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM order_items;
```

In [ ]:
# Archived output only; no computation.

order_id,items
1,[nested value omitted]
2,[nested value omitted]
3,[nested value omitted]


##### 1. Convert all the item names to be UPPERCASE (TRANSFORM Function)

## Learning notes — before source operation

**json-array**: JSON extraction and array operations change structure: parsing creates typed values, while explode can create multiple rows per input. Inspect missing fields, null arrays and empty arrays.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT order_id,
       TRANSFORM(items, x -> UPPER(x)) AS upper_items
  FROM order_items;
```

In [ ]:
# Archived output only; no computation.

order_id,upper_items
1,[nested value omitted]
2,[nested value omitted]
3,[nested value omitted]


##### 2. Filter only items that contains the string 'smart' (FILTER Function)

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT order_id,
       FILTER(items, x -> x LIKE '%smart%') AS smart_items
  FROM order_items;
```

In [ ]:
# Archived output only; no computation.

order_id,smart_items
1,[nested value omitted]
2,[nested value omitted]
3,[nested value omitted]


##### 3. Check to see whether the order includes any 'monitor' (EXISTS Function)

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT order_id,
       EXISTS(items, x -> x = 'monitor') AS has_monitor
  FROM order_items;
```

In [ ]:
# Archived output only; no computation.

order_id,has_monitor
1,True
2,False
3,False


### Array with more than one object

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TEMP VIEW order_items AS
SELECT * FROM VALUES
  (1, array(
        named_struct('name', 'smartphone', 'price', 699),
        named_struct('name', 'laptop', 'price', 1199),
        named_struct('name', 'monitor', 'price', 399)
    )),
  (2, array(
        named_struct('name', 'tablet', 'price', 599),
        named_struct('name', 'headphones', 'price', 199),
        named_struct('name', 'smartwatch', 'price', 299)
    )),
  (3, array(
        named_struct('name', 'keyboard', 'price', 89),
        named_struct('name', 'mouse', 'price', 59)
    ))
AS orders(order_id, items);

```

In [ ]:
# Archived output only; no computation.

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM order_items;
```

In [ ]:
# Archived output only; no computation.

order_id,items
1,[nested value omitted]
2,[nested value omitted]
3,[nested value omitted]


##### 1. Convert all the item names to be UPPERCASE & Add 10% TAX to each item (TRANSFORM Function)

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT order_id,
       TRANSFORM(items, x -> named_struct(
                                          'name', UPPER(x.name),
                                          'price', ROUND(x.price * 1.10, 2)
                                          )) items_with_tax
  FROM order_items
```

In [ ]:
# Archived output only; no computation.

order_id,items_with_tax
1,[nested value omitted]
2,[nested value omitted]
3,[nested value omitted]


##### 2. Calculate the total order amount for each of the order (AGGREGATE Function)

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT order_id,
       AGGREGATE(items, 0, (acc, x) -> acc + x.price) AS total_order_price 
  FROM order_items
```

In [ ]:
# Archived output only; no computation.

order_id,total_order_price
1,2297
2,1097
3,148


### Map Functions

A map is a collection of key-value pairs, like a dictionary  
`{'laptop': 1200, 'phone': 699}`

### Commonly used Higher Order Map Functions  
  - TRANSFORM_VALUES
  - TRANSFORM_KEYS
  - MAP_FILTER

#### Syntax
-----------------------------------------------------------------------------
`<function_name> (map_column, lambda_expression)`  
_lambda expression_: `(key, value) -> expression)`

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TEMP VIEW order_item_prices AS
SELECT * FROM VALUES
  (1, map('smartphone', 699, 'laptop', 1199, 'monitor', 399)),
  (2, map('tablet', 599, 'headphones', 199, 'smartwatch', 299)),
  (3, map('keyboard', 89, 'mouse', 59))
AS orders(order_id, item_prices);
```

In [ ]:
# Archived output only; no computation.

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM order_item_prices;
```

In [ ]:
# Archived output only; no computation.

order_id,item_prices
1,[nested value omitted]
2,[nested value omitted]
3,[nested value omitted]


##### 1. Convert all the item names to be UPPERCASE (TRANSFORM_KEYS Function)

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT order_id,
       TRANSFORM_KEYS(item_prices, (item, price) -> UPPER(item)) AS items_upper_case
  FROM order_item_prices;
```

In [ ]:
# Archived output only; no computation.

order_id,items_upper_case
1,[nested value omitted]
2,[nested value omitted]
3,[nested value omitted]


##### 2. Apply 10% TAX to item prices (TRANSFORM_VALUES Function)

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT order_id,
       TRANSFORM_VALUES(item_prices, (item, price) -> ROUND(price * 1.10, 2)) AS prices_with_tax
  FROM order_item_prices;
```

In [ ]:
# Archived output only; no computation.

order_id,prices_with_tax
1,[nested value omitted]
2,[nested value omitted]
3,[nested value omitted]


##### 3. Filter only items with price above $500 (MAP_FILTER Function)

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT order_id,
       MAP_FILTER(item_prices, (item, price) -> price > 500) AS premium_items
  FROM order_item_prices;
```

In [ ]:
# Archived output only; no computation.

order_id,premium_items
1,[nested value omitted]
2,[nested value omitted]
3,[nested value omitted]
